# Model #

I try to put down a model. I will use the xor-perceptron model as a base and modify that.

first create the lattice, N by N, with Nsol = N**2

I should create a class for the cell object and then the network class is formed by cell objects and when the network is created, each network is associated with a position in the lattice.

Note: for now I'm only considering one possible link between each pair neuron for each direction

In [1]:
import jax
from jax import random as jrd
from jax import numpy as jnp
from jax import debug as jdb

In [ ]:
# parameters of the simulation
par = {'key': jrd.key(1634),    # key for random generation
       'N': 10,
       'int_range': 1}             # interaction range

In [3]:
key = jrd.key(15)
key, subkey = jrd.split(key)
a = jrd.choice(subkey, jnp.arange(100))
a

Array(21, dtype=int32)

In [4]:
from functools import partial
dic = {'a': 0,
       'b': 1}

@partial(jax.jit, static_argnames=['dic'])
def up(dic):
    dic = dict(dic)  # Convert back to a dictionary inside the function
    dic['a'] += 2
    return dic

# Pass the dictionary as a tuple of key-value pairs
dic = up(tuple(dic.items()))

jdb.print('{dic}', dic=dict(dic))

{'a': Array(2, dtype=int32, weak_type=True), 'b': Array(1, dtype=int32, weak_type=True)}


In [5]:
dic = {'a': 0,
       'b': 1}

@jax.jit
def up(dic):
    dic['a'] += 2
    return dic

# Pass the dictionary as a tuple of key-value pairs
dic = up(dic)
dic = up(dic)

jdb.print('{dic}', dic=dict(dic))

{'a': Array(4, dtype=int32, weak_type=True), 'b': Array(1, dtype=int32, weak_type=True)}


In [ ]:
# define a quick function for generating a new random key from par['key'] and update par['key']
def gen_key(par):
    key, subkey = jrd.split(par['key'])     # generate a random key by splitting the key in par
    par['key'] = key                        # update the key in par
    return subkey, par                      # I have to return also par, otherwise, bc of JIT's rules, par['key'] won't update

class Cell:
    
    def __init__(self):     
        self.activation = lambda x: x     # define the activation function through Kolmogorov-Arnold decomposition or Fourier's    # generate a key for random generation 
        self.value = -1    # For now i initialize it to -1, bc I know that can't be 
            
    def generate(self, par):
        self.activation = None  # DEFINE THIS!!!!
        subkey, par = gen_key(par)                  # generate the subkey for the random generation in the following line
        self.value = jrd.choice(subkey, jnp.arange(2)) # Assign an initial value of either 0 or 1   
    
    def compute(self, input):      # compute the value of the cell given the input and the activation. This will be the value sent to the (eventual) other cells.
        self.value = self.activation(jnp.sum(input))
        

class Network:
    
    def __init__(self):             # NON SO SE HA SENSO INIZIALIZZARE COSì LE MATRICI, PERCHè ESSENDO N = 0, VIENE MATRICE = []
        self.N = 0                                                  # N_sol = N**2
        self.lattice = jnp.zeros((self.N,self.N))                   # lattice: N x N - will be a container of Cell objects
        self.J = jnp.ones((self.N,self.N))                          # weight matrix - (initialize to one)
        self.C = jnp.ones((self.N,self.N))                          # connectivity matrix - (initialized to one, so all connected)
        self.B = jnp.zeros((self.N,self.N))                         # bias matrix - (initialize to zero)
        self.fitness = None                                         # Initialize it to None for avoiding recomputation 
        
    def generate(self,par):
        # First set the parameters of the Network from par
        self.N = par['N']
        #self.lattice = jnp.zeros((self.N,self.N))
        self.lattice = [[None] * self.N] * self.N   # I have to use list bc of JAX. this creates an NxN list
        # First generate a cell for each lattice site and assign the former to the latter
        for i in range(self.N):
            for j in range(self.N):
                cell = Cell()                                       # Initialize a Cell object
                cell.generate(par)                                  # Generate it - (also par['key] gets updated here)
                cell.coord = (i,j)                                  # Assign a new attribute 'coord' to the cell object and set it to the coordinates of the lattice site
                self.lattice[(i,j)] = cell                          # Assign the generated cell to the lattice site
        # Then randomly generate weight, bias and connectivity matrices
        subkey, par = gen_key(par)                  # generate the subkey for the random generation in the following line
        self.J = jrd.uniform(subkey,shape=self.J.shape)     # uniformly generate the weights in the weight matrix J
        subkey, par = gen_key(par)                  # generate the subkey for the random generation in the following line
        self.C = jrd.choice(subkey, jnp.arange(2), shape=self.C.shape)  # uniformly generate the links in the connectivity matrix (either 0, no link, or 1, link)       
        subkey, par = gen_key(par)                  # generate the subkey for the random generation in the following line
        self.B = jrd.normal(subkey,shape=self.B.shape)      # extract from a normal distribution centered in 0 with st. dv. = 1 the biases (spero vada bene fatto così)
        # Already compute the fitness of the network
        self.compute_fitness(par)
    
    # PER QUESTE FUNZIONI FACCIO COME HO FATTO PER LO XOR
    def compute_fitness(self, par):
        pass
    
    def ff(self, input:list, verb:int=0):
        # set the value of the two inputs cells through the input value
        self.lattice[0,0].value = input[0]
        self.lattice[self.N,0].value = input[1]
        for i in range(self.N):
            for j in range(self.N): # convention: J_ij = weight of the link FROM cell j TO cell i
                cell = self.lattice[i,j]
                inbound = 0     # value to give in input to the cell, which then computes its value through the activation
                for a in range(self.N):
                    for b in range(self.N):
                        if a != i or b != j:
                            dist = 0.               # compute the distance between cell_ij and cell_ab
                            inbound += self.C[]
                

In [7]:
subkey, par = gen_key(par)                  # generate the subkey for the random generation in the following line
a = jnp.zeros(shape=(3,3))
a = jrd.uniform(subkey, shape=a.shape)
jdb.print('{a}', a=a)
jdb.print('{a.shape}',a=a)
jdb.print('{par}',par=par)

[[0.58271396 0.66091526 0.4762627 ]
 [0.7528697  0.31320405 0.8291925 ]
 [0.7528367  0.31645477 0.59482515]]
(3, 3)
{'N': Array(10, dtype=int32, weak_type=True), 'key': Array((), dtype=key<fry>) overlaying:
[2206521998 2894700812]}


In [8]:
c = Cell()
c.generate(par)
jdb.print('{c}',c=vars(c))

{'activation': None, 'value': Array(0, dtype=int32)}


In [18]:
c = Cell()
d = Cell()
c.generate(par)
d.generate(par)
jdb.print('{c}', c=vars(c))

a = [[None] * 2] * 2  # Use a Python list to store Cell objects
print(a)
a[0] = c
a[1] = d

jdb.print('{a}', a=[vars(cell) for cell in a])  # Convert Cell objects to dictionaries for printing
jdb.print('{c}', c=vars(a[0]))  # No change needed here, as vars(c) is already used
a[0].value = 10

jdb.print('{c}', c=vars(a[0]))

{'activation': None, 'value': Array(0, dtype=int32)}
[[None, None], [None, None]]
[{'activation': None, 'value': Array(0, dtype=int32)}, {'activation': None, 'value': Array(0, dtype=int32)}]
{'activation': None, 'value': Array(0, dtype=int32)}
{'activation': None, 'value': Array(10, dtype=int32, weak_type=True)}


In [ ]:
# Execution
# Notes: put N as a static argname in .jit()
# REMEMBER TO DEAL WITH THE BOUNDARY CONDITIONS ON THE LATTICE